# 02 Cleaning
The pandas cleaning layer (`src/clean.py`) is written independently of the SQL staging layer. This notebook re-runs it, shows every decision with its row impact, and checks it against SQL.

In [1]:
import json, pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
facts = json.load(open("reports/facts.json", encoding="utf-8"))
a = facts["analysis"]
from src import clean
orders, items, log = clean.build()
log

clean: 97,905 orders, 111,752 items, revenue R$13,449,529.68; 10 logged decisions


,step,table,action,rows_before,rows_after,rows_affected,decision
0,1,orders,Drop duplicate order_id,99441,99441,0,Order grain must be unique; keep earliest.
1,2,orders,Keep purchases 2017-01-01 to 2018-08-31,99441,99092,349,Months outside the window are partial (data co...
2,3,orders,Exclude canceled and unavailable orders,99092,97910,1182,These orders generated no revenue; counting th...
3,4,products,Translate two categories missing from the tran...,32951,32951,13,pc_gamer and portateis_cozinha_e_preparadores_...
4,5,products,Label products with no category as 'unknown',32951,32951,610,Kept: their revenue is real; dropping them wou...
5,6,orders,Drop orders with no order items,97910,97905,5,No items means no revenue or category; these a...
6,7,order_reviews,Keep the latest review per order,99224,98673,551,Some orders carry several reviews; the latest ...
7,8,orders,Treat 'delivered' orders with no delivery date...,97905,97905,8,Delivery time and lateness cannot be measured ...
8,9,orders,Flag (not drop) carrier pickup timestamps earl...,97905,97905,166,Timestamp noise in an unused field; customer d...
9,10,order_items,Keep high-price items above the 99.9th percentile,111752,111752,111,"Prices up to R$6,735 are plausible (furniture,..."


## Cross-check against the SQL star schema

In [2]:
from src.warehouse import connect
with connect(read_only=True) as con:
    sql = con.execute('SELECT COUNT(*) AS orders, SUM(revenue) AS revenue FROM fact_orders').df()
py = pd.DataFrame({'orders': [len(orders)], 'revenue': [orders.revenue.sum()]})
pd.concat({'SQL fact_orders': sql, 'Python orders_clean': py})

,,orders,revenue
SQL fact_orders,0,97905,1.344953e+07
Python orders_clean,0,97905,1.344953e+07


In [3]:
orders.head()

,order_id,customer_unique_id,purchase_ts,purchase_month,order_status,customer_state,n_items,revenue,freight,n_sellers,primary_category,payment_total,primary_payment_type,max_installments,is_delivered,delivery_days,delay_days,is_late,review_score,customer_order_seq
0,e22acc9c116caa3f2b7121bbb380d08e,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-01,delivered,SP,1,129.90,12.00,1,bed_bath_table,141.90,credit_card,8,True,6.411227,-5.0,0.0,5.0,1
1,3594e05a005ac4d06a72673270ef9ec9,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-01,delivered,SP,1,18.90,8.29,1,health_beauty,27.19,credit_card,1,True,3.285590,-5.0,0.0,4.0,1
2,b33ec3b699337181488304f362a6b734,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-01,delivered,SC,1,69.00,17.22,1,stationery,86.22,credit_card,8,True,25.731759,-2.0,0.0,3.0,1
3,41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-01,delivered,PA,1,25.99,17.63,1,telephony,43.62,credit_card,4,True,20.037083,-12.0,0.0,4.0,1
4,d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-01,delivered,SP,1,180.00,16.89,1,telephony,196.89,credit_card,6,True,13.141134,-8.0,0.0,5.0,1
